**K-Means clustering** :  Is an unsupervised learning algorithm used to group unlabeled data into a predefined number of clusters (\(k\)).

It works by iteratively assigning each data point to the nearest cluster centroid and then recalculating the centroids as the mean of the new clusters until the assignments stabilize. The goal is to minimize the total distance between each data point and its assigned cluster's centroid.



## Steps

### 1 **Choose k**: The number of clusters k, must be chosen before the algorithm begins.

### 2 . Initialize centroids: The algorithm start by placing `k` initial centroids, which can be randomly selected points in the dataset.

### 3. Assign Data Points: Each data points is assigned to the cluster whose centroid is closest to it, typically using Eucleadian distacnce.

### 4. Update Centroids : After all the points are assigned , the position of each centroid is recalculated to be the mean of all data points within its cluster.

### 4. Repeat:  Steps 3 and 4 are repeated until the centroids no longer move significantly or the cluster assignments no longer change, indicating convergence.



## **Implementation**

In [19]:
import numpy as np

def kmeans(X, k, max_iters=100):
    # X : (n_samples, n_features)
    n_samples, n_features = X.shape

    # Step 1 — initialize centroids by sampling k random points from X
    rng = np.random.default_rng(42)
    centroids = X[rng.choice(n_samples, k, replace=False)]

    for _ in range(max_iters):

        # Step 2 — assign each point to nearest centroid
        # compute the pairwise Eucleadian distances from each data points to each centroid.
        # Assign each point to the closest centroid.
        distances = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        clusters = np.argmin(distances, axis=1)

        # Step 3 — update centroids
        new_centroids = np.array([X[clusters == i].mean(axis=0) for i in range(k)])

        # Stop if converged
        if np.allclose(centroids, new_centroids):
            break

        centroids = new_centroids

    return centroids, clusters


#### Testing

In [20]:
# Create some toy data :

X = np.array([
    [1.0, 2.0], [1.2, 1.8], [0.8, 2.2],
    [6.0, 8.0], [6.4, 7.5], [5.8, 7.8],
])

centers , labels = kmeans(X, k=2)

print("Cluster centers:\n", centers)
print("Labels: ", labels)

Cluster centers:
 [[1.         2.        ]
 [6.06666667 7.76666667]]
Labels:  [0 0 0 1 1 1]


# K-Means ++

Is a improved way to **Choose the initial centroids** before running K-Means.
Insted of picking K random points, it carefully picks centroids so that:
* They are **Far from each other**
* They spread out through the data
* The algorithm converges faster
* Can help avoid bad random starts that create bad clusters



## Intialization

* **Pick the first centroid randomly from the data**
* For each remaining centroid:
  * Comoute the distance of each point to the **nearest** already-chosen centroid,
  * Choose  the next centroid **with Probability proprotional to the squared distance**
    -> Points far away from all chosen centroids are more likely to become new centroids.

  * After K centroids are chosen -> run normal K-Means  
It ensures that centroids start **Spread Out** , so clusters form more naturally.

Thus making
* Lower final loss
* fewer iterations
* More reliable clusters




## Implementation From Scratch

In [31]:
import numpy as np

def kmeans_plus_plus_init(X, k):
  n_samples = X.shape[0]
  # here X.shape() returns a tuple -> (n_samples, n_features)
  # X.shape[0] extracts the First value of the shape tuple -> The number of samples (rows)
  rng = np.random.default_rng(42)
  # create a random number in reproducible way (same as with Generator).

  # Step-1 choose the first centroid randomly
  centroids = [X[rng.integers(n_samples)]]


  for _ in range(1, k):
    # compute the distance from each point to nearest centorid
    dist_sq = np.array([min(np.sum((x - c ) **2) for c in centroids) for x in X])

    # probability proportional to squred distance
    probabilities = dist_sq / dist_sq.sum()

    # Choose the next centroids
    next_centroids = X[rng.choice(n_samples, p=probabilities)]
    centroids.append(next_centroids)

  return np.array(centroids)

def kmeans(X, k, max_iters=100):
  # Intialize using kmeans++  #
  centroids = kmeans_plus_plus_init(X, k)

  for _ in range(max_iters):
    # Assign clusters
    distances  = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
    clusters = np.argmin(distances, axis=1)


    # Update centroids
    new_centroids = np.array([X[clusters == i].mean(axis=0) for i in range(k)])

    # covergence check
    if np.allclose(centroids, new_centroids):
      break

    centroids = new_centroids

  return centroids, clusters





In [32]:
X = np.array([
    [1.0, 2.0], [1.2, 1.8], [0.8, 2.2],
    [6.0, 8.0], [6.4, 7.5], [5.8, 7.8]
])

centers ,labels = kmeans(X, k=2)
print("Clusters centers:\n", centers)
print("Labels: ", labels)

# print(X.shape[0])

Clusters centers:
 [[1.         2.        ]
 [6.06666667 7.76666667]]
Labels:  [0 0 0 1 1 1]


## Accelrated K-Means

Accelerated K-means is a variant of K-Means that runs **faster** that the standard algorithm, without changing the result.

Here we use **Elkan’s K-Means (Triangle Inequality Trick)**
Instead of computing the full distance from every point to every centroid in each iteration, Elkan uses:

Triangle inequality:
  d(x,cj​)≥∣d(ca​,cb​)−d(x,ca​)∣

where
ca	is x's current centroid.

**It helps:
* Reduce the computatin dramatically (sometimes 2x -4x faster)
* Especially effective when clusters are well seperated.
